# Day 5, Sales Brouchers

Creeate a product that builds a Brouchure for a company to be used for prospestive cliente, investors and potential recruiters.

We will be provided a company name and their primary website.


In [ ]:
import os
import json
import sys
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

# Add the parent folder to Python's import path
sys.path.append("../..")

from week1.scraper import (
    fetch_website_links,
    fetch_website_links,
    fetch_website_contents,
)

In [ ]:
load_dotenv(override=True)


api_key = os.getenv("OPENAI_API_KEY")

if api_key and api_key.startswith("sk-proj-") and len(api_key) > 10:
    print("Api key looks so far!")
else:
    print("There's a problem with your API_key, please visit .env")

MODEL = "gpt-5-nano"
openai = OpenAI()

In [ ]:
links = fetch_website_links("https://edwarddonner.com")
links

# First have gpt-5-nano figure out which links are relevant

Use call to gpt-5 nano to read the links on webpage, and respond in structured JSON

it should decide which are relevant and replace relative links such as './about' with "https://coompany.com/about"

this is a better appreacoh because llms because it reqquires nuanced understanding. Imagine doing this without "llms by parsing and analyzing the webpage, it would be very hard!"


In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [ ]:
# We are creting a function for getting links and user prompt


def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.
        
Links (some might be relative links):
    
"""

    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [ ]:
print(get_links_user_prompt("https://edwarddonner.com"))

In [ ]:
def select_relevant_links(url):
    print(f"selecting relevant links for {url} -by using {MODEL}.")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)},
        ],
        response_format={"type": "json_object"},
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"-Found {len(links['links'])} relevant links")
    return links


# def select_relevant_links(url):
#     response = openai.chat.completions.create(
#         model=MODEL,
#         messages=[
#             {"role": "system", "content": link_system_prompt},
#             {"role": "user", "content": get_links_user_prompt(url)},
#         ],
#         response_format={"type": "json_object"},
#     )

#     result = response.choices[0].message.content
#     links = json.loads(result)

#     return links

In [ ]:
select_relevant_links("https://edwarddonner.com")

In [ ]:
select_relevant_links("https://huggingface.co")

# 2nd STEP BROCHURE

- SECOND STEP IS CONVERTING THE FETCHED RELEVANT LINKS IN TO BROCHURE FOROM.


In [ ]:
# from llm_engineering.week2.scraper import fetch_website_contents


def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links["links"]:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [ ]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

In [46]:
# brochure_system_prompt = """"
#     You are an assistant that analyzes the contents of several relevant pages from a company's website  and creates a short brochure about the company for the prospective customers, investors and recruits.
#     Respinse in markdown without code blocks.
#     Include details of company's culture, customers and career/jobs if you have the information.
# """


brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website translate into spanish
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

 

In [49]:
def get_brochure_user_prompt(companu_name, url):
    user_prompt= f""""
        You are looking at a company called {companu_name}  Here are the contents of its landing pages;
        use this information to build a short brochure of the company in markdown without code blocks. \n\n
    """
    
    user_prompt+= fetch_page_and_all_relevant_links(url)
    user_prompt= user_prompt[:5_0000]#tuncate if webpage has lot of content
    return user_prompt

In [50]:
get_brochure_user_prompt("Edward's","https://edwarddonner.com")

selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 12 relevant links


'"\n        You are looking at a company called Edward\'s  Here are the contents of its landing pages;\n        use this information to build a short brochure of the company in markdown without code blocks. \n\n\n    ## Landing Page:\n\nHome - Edward Donner\n\nSkip to content\nAvatar\nCurriculum\nProficiency\nC4\nOutsmart\nAn arena that pits LLMs against each other in a battle of diplomacy and deviousness\nAbout\nPosts\nWell, hi there.\nI’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (\nvery\namateur) and losing myself in\nHacker News\n, nodding my head sagely to things I only half understand.\nI’m the co-founder and CTO of AI startup\nNebula.io\n. I was previously founder and CEO of AI startup untapt,\nacquired in 2021\n, and a Managing Director at JPMorgan.\nI will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and 

# Calling api again based on last call

In [57]:
def create_brochure(company_name, url):
    # print("Generating the brochure")
    response =openai.chat.completions.create(
        model="gpt-4.1-mini" , 
        messages= [{"role": "system" , "content" : brochure_system_prompt}, 
                   {"role": "user" , "content":  get_brochure_user_prompt(company_name,url)}],        
    )
    # print("-Generating the brochure. . .")
    result = response.choices[0].message.content
    print("-Generating the brochure. . .")
    
    display(Markdown(result))

In [59]:
# create_brochure("Edward's Company" , "https://huggingface.co")
create_brochure("Edaward" , "https://edwarddonner.com")


selecting relevant links for https://edwarddonner.com -by using gpt-5-nano.
-Found 6 relevant links
-Generating the brochure. . .


# ¡Conoce a Edaward: Tu Aliado en la Revolución de la Inteligencia Artificial!

---

## ¿Quién está detrás de Edaward?

Hola, soy Ed — un amante de la programación, fanático de experimentar con Modelos de Lenguaje Extensos (LLMs) y cofundador/CTO de la startup de IA **Nebula.io**. Antes de esto, fundé la startup **untapt**, que fue adquirida en 2021, y tuve un paso como Managing Director en JPMorgan. Mi obsesión con la IA me ha llevado a crear cursos en Udemy que ya han conquistado a más de 900,000 estudiantes en 194 países. Sí, ¡casi medio mundo quiere aprender conmigo!

---

## ¿Qué hace Edaward / Nebula.io?

En **Nebula.io**, usamos IA Generativa y machine learning de vanguardia para revolucionar el mundo del reclutamiento. Nuestra tecnología patentada no busca palabras clave; en cambio, ¡encuentra “el match perfecto” entre personas y empleos con una precisión y rapidez que antes parecían imposibles!

¿El sueño final? Ayudar a las personas a descubrir su **Ikigai** — ese propósito vital que hace que el trabajo no solo sea un empleo, sino una fuente de inspiración y realización personal. Porque, seamos honestos, ¿quién quiere ser parte del 77% de gente que no se siente motivada en su trabajo?

---

## La Cultura de Edaward

- **Nerd Alert**: Si sientes que podrías escuchar horas y horas sobre LLMs sin que te duerman los párpados, eres de nuestra tribu.
- **Aprendizaje sin fin**: Con una currícula de cursos que puedes tomar en cualquier orden, pero con un camino natural que te lleva a convertirte en un ingeniero de IA competente y codiciado.
- **Comunidad viva**: Más que cursos, ofrecemos eventos en vivo, charlas que reúnen a miles y una red de ingenieros certificados que brillan en un directorio especial.

---

## Los cursos máster de Edaward (y sí, ¡con cupones en todos!)

1. **AI Builder**: Crea agentes de IA para ti y tus clientes usando herramientas como n8n.
2. **AI Coder**: Programa rápido con agentes que escriben código contigo.
3. **AI Leader**: Cómo liderar con impacto comercial de IA.
4. **AI Engineer Core Track**: Domina la selección y optimización de LLMs.
5. **AI Engineer Agentic Track**: Construye agentes autónomos (sí, robots con cerebro propio).
6. **AI Engineer Production Track**: Despliega tus modelos y agentes a escala sin miedo.

Completar estos seis cursos te hace elegible para el **Directorio de Ingenieros AI Proficientes**, una especie de “LinkedIn VIP para talentos de IA” donde tu gemelo digital charla con futuros empleadores. ¿Mágico? Sí.

---

## Para quién es Edaward

- **Reclutadores** que quieren encontrar talento sin pasar horas en CVs atrapados en palabras clave.
- **Profesionales de IA** que buscan formarse en un currículo completo y reconocido globalmente.
- **Empresas** que desean acelerar su innovación con IA aplicada al reclutamiento y desarrollo de talento.
- **Curiosos y geeks** que quieren aprender y estar a la vanguardia de los avances tecnológicos de la IA.

---

## Burbujas de humor y datos curiosos

- ¿Sabías que Ed tiene una habilidad especial para dormir a sus amigos con sus charlas interminables sobre IA? Por eso decidió hacer cursos en Udemy… para que los fans no tuvieran que aguantar su monólogo 😉
- 900,000 estudiantes, 194 países, 6 cursos… ¿Podrá este curriculum poner fin a la eterna pregunta "¿qué estudiar?" …sí, con IA es posible.

---

## ¿Quieres ser uno de nosotros?

- Checa los cursos, empieza tu aventura IA y súmate al selecto grupo de ingenieros con gemelos digitales que enamoran a reclutadores.
- ¿Tienes dudas existenciales de robótica, IA o el sentido de la vida? El avatar digital de Ed está listo para chatear contigo.
- ¿Eres reclutador? Prueba GRATIS la plataforma y siente la diferencia.

---

## Contacto

- **Web**: [www.edwarddonner.com](http://www.edwarddonner.com)
- **Email**: ed [at] edwarddonner [dot] com
- **Redes Sociales**: LinkedIn | Twitter | Facebook

---

*Edaward: Donde la IA se encuentra con el talento… y se enamoran 💻🤖❤️*

# For chatgpt type, type writer animation
- we gonna use Stream

In [ ]:
def stream_brochure(company_name, url):
    stream =openai.chat.completions.create(
        model="gpt-4.1-mini" , 
        messages= [{"role": "system" , "content" : brochure_system_prompt}, 
                   {"role": "user" , "content":  get_brochure_user_prompt(company_name,url)}],
        stream=True
    )
    
    response= ""
    display_handle=display(Markdown(""), display_id = True)
    for chunk in stream:
        response+=chunk.choices[0].delta.content or '' 
        update_display(Markdown(response),display_id=display_handle.display_id)
    
    

In [ ]:
stream_brochure("Edward" , "https://edwarddonner.com")

# THIRD CALL TO THE LLM
- to convert the brochure into spanish.